# Improvements.ipynb — RAG Experiment Notebook

Systematically improves the baseline RAG pipeline via 4 sequential experiments.
Each experiment uses the best config selected from the previous one.

| Exp | Component | Search space |
|-----|-----------|--------------|
| A | Embedding model | vietnamese-sbert (baseline) · multilingual-e5-large · vietnamese-bi-encoder |
| B | Chunking strategy | 2000/50 (baseline) · 1000/200 · 500/100 |
| C | Retrieval | top-k ∈ {3,5,7,10} · BM25+vector hybrid (RRF) |
| D | Prompt engineering | simple (baseline) · structured output · few-shot |

**Eval protocol:** same 50 fixed pairs (df.head(50)) for all A/B/C/D experiments  
→ apple-to-apple comparison. 100 pairs only for final evaluation (Step 3).


In [ ]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters
!pip install -q faiss-cpu sentence-transformers rank_bm25
!pip install -q pandas numpy tqdm scikit-learn


In [ ]:
import os, re, json, pickle
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_sim

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Constants ──────────────────────────────────────────────────────────
DOCUMENTS_PATH = "Dataset/export_1"
RES_CSV        = "res.csv"
RESULTS_DIR    = "report/results"
N_EVAL         = 50    # fixed for experiments A/B/C/D — do NOT change mid-run
N_FINAL        = 100   # for Step 3 final evaluation only
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Device: {DEVICE}")

# ── Load evaluation data ───────────────────────────────────────────────
df_full  = pd.read_csv(RES_CSV)
df_eval  = df_full.head(N_EVAL).copy().reset_index(drop=True)
df_final = df_full.head(N_FINAL).copy().reset_index(drop=True)
print(f"Eval set  : {len(df_eval)} pairs  (all experiments)")
print(f"Final set : {len(df_final)} pairs (Step 3 only)")

# ── Baseline prompt ────────────────────────────────────────────────────
def prompt_baseline(question, context):
    return "\n".join([
        "Bạn là một trợ lý thông minh. Trả lời câu hỏi dựa trên ngữ cảnh sau.",
        "Nếu không đủ thông tin, hãy nói rõ điều đó, không được tự bịa đặt.",
        "",
        "Ngữ cảnh:",
        context,
        "",
        f"Câu hỏi: {question}",
        "Trả lời ngắn gọn và chính xác nhất có thể:",
    ])

# ── Text normalization ─────────────────────────────────────────────────
def normalize_text(text):
    text = str(text).lower().replace("\n", " ").strip()
    text = re.sub(r'[^\w\s]', '', text)
    return re.sub(r'\s+', ' ', text)

# ── Metric functions ───────────────────────────────────────────────────
def cosine_sim_score(pred, true, emb_model):
    if not pred or not true:
        return 0.0
    return float(sk_cosine_sim([emb_model.embed_query(pred)],
                               [emb_model.embed_query(true)])[0][0])

def jaccard(pred, true):
    p = set(normalize_text(pred).split())
    t = set(normalize_text(true).split())
    return len(p & t) / len(p | t) if p and t else 0.0

def token_overlap(pred, true):
    p = normalize_text(pred).split()
    t = normalize_text(true).split()
    return sum((Counter(p) & Counter(t)).values()) / len(t) if t else 0.0

def bleu(pred, true):
    p = normalize_text(pred).split()
    t = normalize_text(true).split()
    if not p or not t:
        return 0.0
    ov   = sum((Counter(p) & Counter(t)).values())
    prec = ov / len(p)
    bp   = 1.0 if len(p) >= len(t) else np.exp(1 - len(t) / len(p))
    return bp * prec

def rouge_l(pred, true):
    p = normalize_text(pred).split()
    t = normalize_text(true).split()
    if not p or not t:
        return 0.0
    m, n = len(p), len(t)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[i-1] == t[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs  = dp[m][n]
    prec, rec = lcs / m, lcs / n
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def compute_metrics(pred, true, emb_model):
    return {
        "Cosine_Similarity": cosine_sim_score(pred, true, emb_model),
        "Jaccard"          : jaccard(pred, true),
        "Token_Overlap"    : token_overlap(pred, true),
        "BLEU"             : bleu(pred, true),
        "ROUGE_L"          : rouge_l(pred, true),
    }

# ── Core evaluation function ───────────────────────────────────────────
def evaluate_config(vectorstore, emb_model, llm, df, k=5,
                    prompt_fn=prompt_baseline, hybrid_fn=None, label=""):
    # hybrid_fn: optional fn(question, k) -> list[str] replacing vectorstore search
    rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Eval [{label}]"):
        question     = str(row["Câu Hỏi"]).strip()
        ground_truth = str(row["Trả lời"]).strip()
        try:
            if hybrid_fn is not None:
                context_texts = hybrid_fn(question, k)
            else:
                docs = vectorstore.similarity_search(question, k=k)
                context_texts = [d.page_content for d in docs]
            context  = "\n\n".join(context_texts)
            prompt   = prompt_fn(question, context)
            response = llm.invoke(prompt)
            response = response.strip() if isinstance(response, str) else str(response).strip()
        except Exception as e:
            print(f"  [!] row {idx}: {e}")
            response = ""
        rows.append(compute_metrics(response, ground_truth, emb_model))
    means = pd.DataFrame(rows).mean().to_dict()
    means["label"] = label
    return means

print("Setup complete.")


In [ ]:
LLM_MODEL_ID = "Qwen/Qwen1.5-1.8B"
print(f"Loading LLM: {LLM_MODEL_ID} ...")

llm = HuggingFacePipeline.from_model_id(
    model_id=LLM_MODEL_ID,
    task="text-generation",
    device_map="auto",
    model_kwargs={"torch_dtype": torch.float16},
    pipeline_kwargs={
        "max_new_tokens" : 512,
        "do_sample"      : False,
        "return_full_text": False,   # return only generated text, NOT the prompt
    },
)
print("LLM ready.")


## Experiment A — Embedding Model

Compare three Vietnamese embedding models. Rebuild FAISS index for each.
Use baseline FAISS (`faiss_db/`) for A1 if it already exists.

> **Note:** `multilingual-e5-large` requires `"query: "` / `"passage: "` prefixes.
> `load_embedding_model()` handles this automatically via subclassing.


In [ ]:
def load_embedding_model(model_name):
    """Load HuggingFaceEmbeddings; adds e5 prefixes if needed."""
    is_e5 = "e5" in model_name.lower()
    if is_e5:
        class _E5Embed(HuggingFaceEmbeddings):
            def embed_query(self, text):
                return super().embed_query("query: " + text)
            def embed_documents(self, texts):
                return super().embed_documents(["passage: " + t for t in texts])
        cls = _E5Embed
    else:
        cls = HuggingFaceEmbeddings
    return cls(
        model_name=model_name,
        model_kwargs={"device": DEVICE},
    )


def build_or_load_faiss(emb_model, model_name, chunk_size=2000, chunk_overlap=50):
    """Build FAISS or load from cache. Returns vectorstore."""
    safe     = model_name.split("/")[-1]
    faiss_dir = f"faiss_db_{safe}_{chunk_size}_{chunk_overlap}"

    # Reuse baseline index (built by Baseline.ipynb) if applicable
    if model_name == "keepitreal/vietnamese-sbert" and chunk_size == 2000 and chunk_overlap == 50:
        if os.path.exists("faiss_db"):
            print("Loading baseline FAISS (faiss_db/) ...")
            return FAISS.load_local("faiss_db", emb_model, allow_dangerous_deserialization=True)

    if os.path.exists(faiss_dir):
        print(f"Loading cached FAISS '{faiss_dir}' ...")
        return FAISS.load_local(faiss_dir, emb_model, allow_dangerous_deserialization=True)

    # Build from scratch
    print(f"Building FAISS | {model_name} | chunk={chunk_size}/{chunk_overlap} ...")
    splitter   = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    all_chunks = []
    files      = sorted(f for f in os.listdir(DOCUMENTS_PATH) if f.endswith(".txt"))
    for fname in tqdm(files, desc="Splitting docs"):
        try:
            content = open(os.path.join(DOCUMENTS_PATH, fname),
                           encoding="utf-8", errors="ignore").read()
            all_chunks.extend(splitter.create_documents([content]))
        except Exception:
            pass
    print(f"  {len(all_chunks)} chunks from {len(files)} docs")

    BATCH = 512
    vs = None
    for i in tqdm(range(0, len(all_chunks), BATCH), desc="Embedding"):
        batch = all_chunks[i:i + BATCH]
        if vs is None:
            vs = FAISS.from_documents(batch, emb_model)
        else:
            vs.add_documents(batch)
    vs.save_local(faiss_dir)
    print(f"  Saved to '{faiss_dir}'")
    return vs

print("Helpers ready.")


In [ ]:
# A1: keepitreal/vietnamese-sbert (baseline)
MODEL_A1 = "keepitreal/vietnamese-sbert"
emb_A1   = load_embedding_model(MODEL_A1)
vs_A1    = build_or_load_faiss(emb_A1, MODEL_A1)
res_A1   = evaluate_config(vs_A1, emb_A1, llm, df_eval, k=5, label="A1-sbert")
print(res_A1)


In [ ]:
# A2: intfloat/multilingual-e5-large  (e5 prefixes handled by load_embedding_model)
MODEL_A2 = "intfloat/multilingual-e5-large"
emb_A2   = load_embedding_model(MODEL_A2)
vs_A2    = build_or_load_faiss(emb_A2, MODEL_A2)
res_A2   = evaluate_config(vs_A2, emb_A2, llm, df_eval, k=5, label="A2-e5-large")
print(res_A2)


In [ ]:
# A3: bkai-foundation-models/vietnamese-bi-encoder
MODEL_A3 = "bkai-foundation-models/vietnamese-bi-encoder"
emb_A3   = load_embedding_model(MODEL_A3)
vs_A3    = build_or_load_faiss(emb_A3, MODEL_A3)
res_A3   = evaluate_config(vs_A3, emb_A3, llm, df_eval, k=5, label="A3-bi-encoder")
print(res_A3)


In [ ]:
# Exp A — Summary
_A_map = {
    "A1-sbert"     : (MODEL_A1, emb_A1, vs_A1),
    "A2-e5-large"  : (MODEL_A2, emb_A2, vs_A2),
    "A3-bi-encoder": (MODEL_A3, emb_A3, vs_A3),
}
df_A = pd.DataFrame([res_A1, res_A2, res_A3]).set_index("label")
print("\n=== Experiment A — Embedding Model Results ===")
print(df_A.round(4).to_string())
df_A.to_csv(f"{RESULTS_DIR}/exp_A_embedding.csv")

# Auto-select by ROUGE_L; override manually if needed
best_A_lbl          = df_A["ROUGE_L"].idxmax()
BEST_EMBEDDING_NAME = _A_map[best_A_lbl][0]
BEST_EMB_MODEL      = _A_map[best_A_lbl][1]
BEST_VS_2000        = _A_map[best_A_lbl][2]   # chunk 2000/50 with best embedding
print(f"\n✓ Best embedding: {best_A_lbl}  ({BEST_EMBEDDING_NAME})")
print("  Override: reassign BEST_EMBEDDING_NAME / BEST_EMB_MODEL / BEST_VS_2000 if needed.")


## Experiment B — Chunking Strategy

Using best embedding from Exp A. Each config builds a separate FAISS index
(saved to `faiss_db_<model>_<size>_<overlap>/`) to enable re-runs without rebuilding.

> **Note:** B1 reuses the vectorstore already built in Exp A (same model + 2000/50 config).


In [ ]:
# B1: chunk 2000/50 (baseline) — reuse BEST_VS_2000 from Exp A
res_B1 = evaluate_config(BEST_VS_2000, BEST_EMB_MODEL, llm, df_eval, k=5, label="B1-2000-50")
print(res_B1)


In [ ]:
# B2: chunk 1000/200
vs_B2  = build_or_load_faiss(BEST_EMB_MODEL, BEST_EMBEDDING_NAME, chunk_size=1000, chunk_overlap=200)
res_B2 = evaluate_config(vs_B2, BEST_EMB_MODEL, llm, df_eval, k=5, label="B2-1000-200")
print(res_B2)


In [ ]:
# B3: chunk 500/100
vs_B3  = build_or_load_faiss(BEST_EMB_MODEL, BEST_EMBEDDING_NAME, chunk_size=500, chunk_overlap=100)
res_B3 = evaluate_config(vs_B3, BEST_EMB_MODEL, llm, df_eval, k=5, label="B3-500-100")
print(res_B3)


In [ ]:
# Exp B — Summary
_B_map = {
    "B1-2000-50" : (BEST_VS_2000, 2000, 50),
    "B2-1000-200": (vs_B2,        1000, 200),
    "B3-500-100" : (vs_B3,        500,  100),
}
df_B = pd.DataFrame([res_B1, res_B2, res_B3]).set_index("label")
print("\n=== Experiment B — Chunking Strategy Results ===")
print(df_B.round(4).to_string())
df_B.to_csv(f"{RESULTS_DIR}/exp_B_chunking.csv")

best_B_lbl         = df_B["ROUGE_L"].idxmax()
BEST_VS            = _B_map[best_B_lbl][0]
BEST_CHUNK_SIZE    = _B_map[best_B_lbl][1]
BEST_CHUNK_OVERLAP = _B_map[best_B_lbl][2]
print(f"\n✓ Best chunking: {best_B_lbl}  (size={BEST_CHUNK_SIZE}, overlap={BEST_CHUNK_OVERLAP})")


## Experiment C — Retrieval Strategy

**C1:** Top-k tuning — no FAISS rebuild needed, just change `k`.

**C2:** BM25 + vector hybrid via Reciprocal Rank Fusion (RRF).
Corpus is rebuilt from documents using best chunk config (cached as `.pkl`).


In [ ]:
# Exp C1 — Top-k tuning
exp_C1_rows = []
for k_val in [3, 5, 7, 10]:
    res = evaluate_config(BEST_VS, BEST_EMB_MODEL, llm, df_eval,
                          k=k_val, label=f"C1-k{k_val}")
    exp_C1_rows.append(res)
    print(f"  k={k_val:2d}  ROUGE_L={res['ROUGE_L']:.4f}  Cosine={res['Cosine_Similarity']:.4f}")

df_C1    = pd.DataFrame(exp_C1_rows).set_index("label")
BEST_K   = int(df_C1["ROUGE_L"].idxmax().split("k")[-1])
print(f"\n✓ Best k = {BEST_K}")


In [ ]:
from rank_bm25 import BM25Okapi

# Build (or load cached) text corpus for best chunk config
corpus_pkl = f"bm25_corpus_{BEST_CHUNK_SIZE}_{BEST_CHUNK_OVERLAP}.pkl"

if os.path.exists(corpus_pkl):
    CORPUS_TEXTS = pickle.load(open(corpus_pkl, "rb"))
    print(f"Loaded BM25 corpus: {len(CORPUS_TEXTS)} chunks")
else:
    print(f"Building corpus (chunk={BEST_CHUNK_SIZE}/{BEST_CHUNK_OVERLAP}) ...")
    splitter     = RecursiveCharacterTextSplitter(
        chunk_size=BEST_CHUNK_SIZE, chunk_overlap=BEST_CHUNK_OVERLAP)
    CORPUS_TEXTS = []
    files        = sorted(f for f in os.listdir(DOCUMENTS_PATH) if f.endswith(".txt"))
    for fname in tqdm(files, desc="Loading docs"):
        try:
            content = open(os.path.join(DOCUMENTS_PATH, fname),
                           encoding="utf-8", errors="ignore").read()
            CORPUS_TEXTS.extend(
                c.page_content for c in splitter.create_documents([content]))
        except Exception:
            pass
    pickle.dump(CORPUS_TEXTS, open(corpus_pkl, "wb"))
    print(f"Corpus: {len(CORPUS_TEXTS)} chunks saved to {corpus_pkl}")

# BM25 index (fast to build, ~30-60s — no caching needed)
print("Building BM25 index ...")
BM25_INDEX = BM25Okapi([t.split() for t in CORPUS_TEXTS])
print("BM25 ready.")


In [ ]:
# Exp C2 — BM25 + vector hybrid via Reciprocal Rank Fusion (RRF)
def rrf_hybrid_fn(question, k, rrf_k=60):
    # Vector results (over-retrieve, then fuse)
    vec_docs   = BEST_VS.similarity_search(question, k=k * 2)
    vec_ranked = [d.page_content for d in vec_docs]

    # BM25 results
    scores     = BM25_INDEX.get_scores(question.split())
    top_idx    = np.argsort(scores)[::-1][:k * 2]
    bm25_ranked = [CORPUS_TEXTS[i] for i in top_idx]

    # RRF fusion
    rrf_scores = {}
    for rank, text in enumerate(vec_ranked):
        rrf_scores[text] = rrf_scores.get(text, 0) + 1.0 / (rrf_k + rank + 1)
    for rank, text in enumerate(bm25_ranked):
        rrf_scores[text] = rrf_scores.get(text, 0) + 1.0 / (rrf_k + rank + 1)
    return sorted(rrf_scores, key=lambda x: -rrf_scores[x])[:k]

res_C2 = evaluate_config(
    BEST_VS, BEST_EMB_MODEL, llm, df_eval,
    k=BEST_K, hybrid_fn=rrf_hybrid_fn, label="C2-bm25-hybrid",
)
print(res_C2)


In [ ]:
# Exp C — Summary
all_C_rows = exp_C1_rows + [res_C2]
df_C       = pd.DataFrame(all_C_rows).set_index("label")
print("\n=== Experiment C — Retrieval Strategy Results ===")
print(df_C.round(4).to_string())
df_C.to_csv(f"{RESULTS_DIR}/exp_C_retrieval.csv")

best_C_lbl = df_C["ROUGE_L"].idxmax()
print(f"\n✓ Best retrieval: {best_C_lbl}")

if best_C_lbl == "C2-bm25-hybrid":
    BEST_RETRIEVAL_FN = rrf_hybrid_fn
    print("  → Using BM25 hybrid retrieval")
else:
    BEST_K            = int(best_C_lbl.split('k')[-1])
    BEST_RETRIEVAL_FN = None   # use vectorstore directly
    print(f"  → Using vector search k={BEST_K}")


## Experiment D — Prompt Engineering

Three prompt variants tested on the best config from A+B+C.
All variants use the same retrieval; only the prompt template changes.

| ID | Variant | Strategy |
|----|---------|----------|
| D1 | Baseline | Simple Vietnamese instruction (same as Baseline.ipynb) |
| D2 | Structured output | Force `[Căn cứ pháp lý]` + `[Nội dung]` format |
| D3 | Few-shot | One in-context Q&A example (pair #51, outside eval set) |

> **Token budget note:** few-shot adds ~200-300 tokens to each prompt.
> With `max_new_tokens=512`, total generation stays within Qwen1.5-1.8B limits.


In [ ]:
# Few-shot example: use pair #N_EVAL+1 (outside df_eval to avoid leakage)
fs_row = df_full.iloc[N_EVAL]
FS_Q   = str(fs_row["Câu Hỏi"]).strip()
FS_A   = str(fs_row["Trả lời"]).strip()[:300]   # truncate to stay within token budget

def prompt_structured(question, context):
    return "\n".join([
        "Bạn là trợ lý pháp lý. Trả lời dựa trên văn bản pháp luật dưới đây.",
        "Chỉ dùng thông tin trong ngữ cảnh. Không bịa đặt.",
        "",
        "Ngữ cảnh:",
        context,
        "",
        f"Câu hỏi: {question}",
        "",
        "Trả lời theo format:",
        "[Căn cứ pháp lý]: <tên văn bản/điều khoản liên quan>",
        "[Nội dung]: <câu trả lời chi tiết>",
    ])

def prompt_fewshot(question, context):
    return "\n".join([
        "Bạn là trợ lý pháp lý. Trả lời câu hỏi dựa trên ngữ cảnh được cung cấp.",
        "",
        "Ví dụ:",
        f"Câu hỏi: {FS_Q}",
        f"Trả lời: {FS_A}",
        "",
        "---",
        "Ngữ cảnh:",
        context,
        "",
        f"Câu hỏi: {question}",
        "Trả lời:",
    ])

# Evaluate all prompt variants
_PROMPTS = [
    (prompt_baseline,   "D1-baseline"),
    (prompt_structured, "D2-structured"),
    (prompt_fewshot,    "D3-fewshot"),
]
exp_D_rows = []
for prompt_fn, lbl in _PROMPTS:
    res = evaluate_config(
        BEST_VS, BEST_EMB_MODEL, llm, df_eval,
        k=BEST_K, prompt_fn=prompt_fn,
        hybrid_fn=BEST_RETRIEVAL_FN, label=lbl,
    )
    exp_D_rows.append(res)
    print(f"  {lbl:15s}  ROUGE_L={res['ROUGE_L']:.4f}  Cosine={res['Cosine_Similarity']:.4f}")


In [ ]:
# Exp D — Summary
_D_prompt_map = {
    "D1-baseline"  : prompt_baseline,
    "D2-structured": prompt_structured,
    "D3-fewshot"   : prompt_fewshot,
}
df_D = pd.DataFrame(exp_D_rows).set_index("label")
print("\n=== Experiment D — Prompt Engineering Results ===")
print(df_D.round(4).to_string())
df_D.to_csv(f"{RESULTS_DIR}/exp_D_prompt.csv")

best_D_lbl    = df_D["ROUGE_L"].idxmax()
BEST_PROMPT_FN = _D_prompt_map[best_D_lbl]
print(f"\n✓ Best prompt: {best_D_lbl}")


## Final Comparison & Step 3 — Full Evaluation (100 pairs)

Re-run baseline and best config on 100 pairs for the official report numbers.

**Best config selected:**
- Embedding: `BEST_EMBEDDING_NAME`
- Chunk: `BEST_CHUNK_SIZE` / `BEST_CHUNK_OVERLAP`
- k: `BEST_K`
- Retrieval: `BEST_RETRIEVAL_FN` (None = vector only)
- Prompt: `BEST_PROMPT_FN.__name__`


In [ ]:
# Print best config for verification before running the expensive final eval
print("=== Best Configuration ===")
print(f"  Embedding  : {BEST_EMBEDDING_NAME}")
print(f"  Chunk      : {BEST_CHUNK_SIZE} / {BEST_CHUNK_OVERLAP}")
print(f"  Top-k      : {BEST_K}")
print(f"  Retrieval  : {'BM25 hybrid (RRF)' if BEST_RETRIEVAL_FN else 'vector only'}")
print(f"  Prompt     : {BEST_PROMPT_FN.__name__}")

# Override any selection manually before running the next cell if needed:
# BEST_EMBEDDING_NAME = "keepitreal/vietnamese-sbert"
# BEST_EMB_MODEL      = emb_A1
# BEST_VS             = vs_A1
# BEST_CHUNK_SIZE     = 2000
# BEST_CHUNK_OVERLAP  = 50
# BEST_K              = 5
# BEST_RETRIEVAL_FN   = None
# BEST_PROMPT_FN      = prompt_baseline


In [ ]:
METRICS_COLS = ["Cosine_Similarity", "Jaccard", "Token_Overlap", "BLEU", "ROUGE_L"]

print("Running BASELINE on 100 pairs ...")
res_baseline_100 = evaluate_config(
    vs_A1, emb_A1, llm, df_final,
    k=5, prompt_fn=prompt_baseline,
    hybrid_fn=None, label="BASELINE-100",
)

print("Running BEST CONFIG on 100 pairs ...")
res_best_100 = evaluate_config(
    BEST_VS, BEST_EMB_MODEL, llm, df_final,
    k=BEST_K, prompt_fn=BEST_PROMPT_FN,
    hybrid_fn=BEST_RETRIEVAL_FN, label="BEST-100",
)

# ── Results table ──────────────────────────────────────────────────────
df_final_res = pd.DataFrame([res_baseline_100, res_best_100]).set_index("label")
print("\n=== FINAL RESULTS (100 pairs) ===")
print(df_final_res[METRICS_COLS].round(4).to_string())

print("\n=== IMPROVEMENT DELTA ===")
for col in METRICS_COLS:
    base  = df_final_res.loc["BASELINE-100", col]
    best  = df_final_res.loc["BEST-100", col]
    delta = best - base
    pct   = delta / base * 100 if base > 0 else float('inf')
    print(f"  {col:20s}: {base:.4f} -> {best:.4f}  ({delta:+.4f}, {pct:+.1f}%)")

# ── Save all results ───────────────────────────────────────────────────
df_final_res.to_csv(f"{RESULTS_DIR}/final_evaluation_100pairs.csv")

all_rows = (exp_C1_rows + [res_C2] + exp_D_rows +
            [res_baseline_100, res_best_100])
df_all = pd.DataFrame(
    [res_A1, res_A2, res_A3,
     res_B1, res_B2, res_B3]
    + all_rows
).set_index("label")
df_all.to_csv(f"{RESULTS_DIR}/comparison_table.csv")
print(f"\nAll results saved to {RESULTS_DIR}/")
print("  exp_A_embedding.csv")
print("  exp_B_chunking.csv")
print("  exp_C_retrieval.csv")
print("  exp_D_prompt.csv")
print("  final_evaluation_100pairs.csv")
print("  comparison_table.csv")
